# 03 — Wearable-Augmented Risk Model
XGBoost with clinical + wearable + SDOH features. Equity-aware imputation.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("pcp-panel-intelligence"):
        os.system("git clone https://github.com/Hannah-Hiltz/pcp-panel-intelligence.git")
    os.chdir("pcp-panel-intelligence")
    sys.path.insert(0, "src")
    os.system("pip install -q xgboost shap scikit-learn joblib")
    DATA_DIR  = "data/raw";      PROC_DIR  = "data/processed"
    FIG_DIR   = "reports/figures"; MODEL_DIR = "models"
    WEAR_DIR  = "data/raw/wearables"; PANEL_DIR = "data/raw/panel"
else:
    sys.path.insert(0, "../src")
    DATA_DIR  = "../data/raw";   PROC_DIR  = "../data/processed"
    FIG_DIR   = "../reports/figures"; MODEL_DIR = "../models"
    WEAR_DIR  = "../data/raw/wearables"; PANEL_DIR = "../data/raw/panel"

for d in [FIG_DIR, PROC_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print(f"Environment: {'Colab' if IN_COLAB else 'Jupyter'}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, average_precision_score
from xgboost import XGBClassifier
import shap, joblib

df = pd.read_csv(f"{PANEL_DIR}/patient_panel_flat.csv")
print(f"Panel: {len(df):,} patients")


## Feature sets — clinical, wearable, SDOH

In [ ]:
CLINICAL_FEATURES = [
    "age","n_conditions","n_quality_gaps","days_since_pcp_visit",
    "er_visits_12m","hospitalizations_12m",
    "HbA1c","SBP","DBP","LDL","eGFR","BMI",
    "has_Type_2_Diabetes","has_Hypertension","has_Hyperlipidemia",
    "has_Coronary_Artery_Disease","has_Heart_Failure","has_COPD",
    "has_CKD","has_Depression","has_Obesity","has_Atrial_Fibrillation",
]
WEARABLE_FEATURES = [
    "wear_steps_7d_avg","wear_steps_trend","wear_steps_baseline_dev",
    "wear_rhr_7d_avg","wear_rhr_trend","wear_rhr_baseline_dev",
    "wear_hrv_7d_avg","wear_hrv_trend",
    "wear_sleep_7d_avg","wear_active_7d_avg",
    "wear_anomaly_flag","wear_completeness",
]
SDOH_FEATURES = [
    "zip_income_quintile","food_insecurity","housing_instability","transportation_barrier",
]
CAT_FEATURES = ["gender","race_ethnicity","insurance"]

ALL_NUMERIC = CLINICAL_FEATURES + WEARABLE_FEATURES + SDOH_FEATURES
available   = [c for c in ALL_NUMERIC if c in df.columns]
X = df[available + CAT_FEATURES]
y = df["outreach_priority"].isin(["Critical","High"]).astype(int)

print(f"Features: {len(available)} numeric + {len(CAT_FEATURES)} categorical")
print(f"Target: {y.sum()} High/Critical ({y.mean():.1%})")
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


## Build and train model

In [ ]:
num_pipe = Pipeline([("imp",SimpleImputer(strategy="median")),("scl",StandardScaler())])
cat_pipe = Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("enc",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
pre = ColumnTransformer([("num",num_pipe,available),("cat",cat_pipe,CAT_FEATURES)],remainder="drop")

model = Pipeline([
    ("preprocessor", pre),
    ("classifier", XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=3, eval_metric="auc",
        random_state=42, n_jobs=-1,
    )),
])
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:,1]
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.3f}")
print(f"Avg Precision: {average_precision_score(y_test, y_prob):.3f}")
print(classification_report(y_test, model.predict(X_test), target_names=["Routine/Moderate","High/Critical"]))


## SHAP — wearable features vs clinical features

In [ ]:
X_tf = model.named_steps["preprocessor"].transform(X_test)
clf  = model.named_steps["classifier"]
cat_names = model.named_steps["preprocessor"].named_transformers_["cat"]["enc"]    .get_feature_names_out(CAT_FEATURES).tolist()
feature_names = available + cat_names

explainer   = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_tf)

shap.summary_plot(shap_values, X_tf, feature_names=feature_names, max_display=20, show=False)
plt.title("SHAP — clinical + wearable + SDOH feature importance")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/shap_full.png", dpi=150, bbox_inches="tight"); plt.show()


## Wearable feature contribution — isolated

In [ ]:
wear_idx = [i for i, n in enumerate(feature_names) if n in WEARABLE_FEATURES]
if wear_idx:
    wear_shap = np.abs(shap_values[:, wear_idx]).mean(axis=0)
    wear_names = [feature_names[i] for i in wear_idx]
    sorted_idx = np.argsort(wear_shap)[::-1][:10]
    fig, ax = plt.subplots(figsize=(10,5))
    ax.barh([wear_names[i] for i in sorted_idx], [wear_shap[i] for i in sorted_idx], color="#378ADD")
    ax.set_title("Mean |SHAP| — wearable features only", fontsize=13)
    ax.set_xlabel("Mean absolute SHAP value")
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/shap_wearable.png", dpi=150, bbox_inches="tight"); plt.show()


## Equity audit — model performance by income quintile

In [ ]:
test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df["y_prob"] = y_prob
test_df["zip_income_quintile"] = df.loc[y_test.index, "zip_income_quintile"].values

print("Model AUC by income quintile:")
for q in sorted(test_df["zip_income_quintile"].unique()):
    sub = test_df[test_df["zip_income_quintile"]==q]
    if sub["y_true"].nunique() > 1:
        auc = roc_auc_score(sub["y_true"], sub["y_prob"])
        print(f"  Q{q}: AUC={auc:.3f} (n={len(sub)})")

print("\nKey: If AUC drops for Q1/Q2, the model is less reliable for")
print("lower-income patients — likely due to missing wearable data.")
print("Conservative-upward imputation is designed to mitigate this.")


## Save model

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, f"{MODEL_DIR}/risk_model_wearable.joblib")

scored = df.copy()
scored["risk_probability"] = model.predict_proba(df[[c for c in available+CAT_FEATURES if c in df.columns]])[:,1]
scored["risk_tier"] = pd.cut(scored["risk_probability"], bins=[0,0.3,0.5,0.7,1.0],
                              labels=["Low","Moderate","High","Critical"])
scored.sort_values("risk_probability", ascending=False).to_csv(f"{PROC_DIR}/scored_panel_wearable.csv", index=False)
print("Model saved.")
